<a href="https://colab.research.google.com/github/arups330/ElitLab_MED_VQA/blob/main/Performance_with_CoT_v3(abdomen_Open).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# ---------------------------------------------------------------------------
# 0. Install dependencies
# ---------------------------------------------------------------------------
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate
!pip install -q "pillow<11"
!pip install -q rouge-score bert-score nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.9 MB/s eta 0:00:00


In [ ]:
import sys
sys.modules["torchao"] = None   # avoid the earlier torch.int1 crash

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

import os
import gc
import json
import glob
import zipfile
import torch
import pandas as pd
from PIL import Image

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

_smoothing = SmoothingFunction().method1
_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

In [ ]:
# ---------------------------------------------------------------------------
# 1. Choose which dataset to run this session. Only this dataset's zip is
#    extracted and only its open.csv rows are used below.
# ---------------------------------------------------------------------------
DATASET = "Abdomen"   # <-- change to another key for a different run (then RESTART RUNTIME)

assert DATASET in ("Abdomen",), "DATASET must be 'Abdomen'"

ZIP_FILES = {
    "Abdomen": "/content/Abdomen_With_CoT(test).zip",
}

import zipfile

zip_path = ZIP_FILES[DATASET]
extract_dir = f"/content/{DATASET}_Test_with_CoT"

if not os.path.exists(extract_dir):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
else:
    print(f"Already extracted: {extract_dir}")

TEST_DATA_ROOT = extract_dir
print("\nTEST_DATA_ROOT =", TEST_DATA_ROOT)
print(f"\nContents of {TEST_DATA_ROOT}:")
print(" ", os.listdir(TEST_DATA_ROOT))




Extracting /content/Abdomen_With_CoT(test).zip...

TEST_DATA_ROOT = /content/Abdomen_Test_with_CoT

Contents of /content/Abdomen_Test_with_CoT:
  ['test']


In [ ]:
# ---------------------------------------------------------------------------
# MODIFIED: Path to the CSV inside the zip -- swapped from
# closed_with_CoT.csv to Open_with_CoT.csv (same directory).
# ---------------------------------------------------------------------------
CSV_PATH = os.path.join(TEST_DATA_ROOT, "test", "Open_with_CoT.csv")
print("\nCSV_PATH =", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

df = pd.read_csv(CSV_PATH)
print("\nLoaded CSV shape:", df.shape)
df.head()


CSV_PATH = /content/Abdomen_Test_with_CoT/test/Open_with_CoT.csv
Exists: True

Loaded CSV shape: (153, 14)


,image_file,img_id,location,modality,question,answer,q_lang,answer_type,content_type,base_type,qid,triple,split_dir,CoT
0,xmlab103_source.jpg,103,Abdomen,CT,What modality is used to take this image?,CT,en,OPEN,Modality,vqa,11945,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
1,xmlab103_source.jpg,103,Abdomen,CT,Which part of the body does this image belong to?,Chest,en,OPEN,Position,vqa,11946,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
2,xmlab103_source.jpg,103,Abdomen,CT,What is the main organ in the image?,Lung,en,OPEN,Organ,vqa,11947,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
3,xmlab103_source.jpg,103,Abdomen,CT,What is the largest organ in the picture?,Lung,en,OPEN,Size,vqa,11948,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
4,xmlab103_source.jpg,103,Abdomen,CT,What diseases are included in the picture?,Lung Cancer,en,OPEN,Abnormality,vqa,11951,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...


In [ ]:
# Optional: set a maximum number of rows to process for each model to speed up testing.
MAX_ROWS_PER_MODEL = None # e.g. 50

open_csvs = glob.glob(os.path.join(TEST_DATA_ROOT, "**", "Open_with_CoT.csv"), recursive=True)

print(f"\nFound {len(open_csvs)} Open_with_CoT.csv test files for Abdomen:")
for p in open_csvs:
    print(" ", p)

if not open_csvs:
    raise FileNotFoundError(
        f"No Open_with_CoT.csv found under {TEST_DATA_ROOT} -- check ZIP_PATH "
        f"and confirm the exact filename/casing with: "
        f"os.listdir(os.path.join(TEST_DATA_ROOT, 'test'))"
    )

open_frames = []
for csv_path in open_csvs:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    open_frames.append(df)

open_df = pd.concat(open_frames, ignore_index=True)

COT_COL = "CoT" if "CoT" in open_df.columns else "cot"
open_df = open_df[open_df[COT_COL].notna() & (open_df[COT_COL].str.strip() != "")]
if MAX_ROWS_PER_MODEL:
    open_df = open_df.head(MAX_ROWS_PER_MODEL)
print(f"\nTotal open-ended test rows for Abdomen (with valid CoT): {len(open_df)}")

OPEN_IMG_COL = "image_file" if "image_file" in open_df.columns else "img_name"

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    for c in [os.path.join(split_dir, str(img_name)), os.path.join(split_dir, flat_name),
              os.path.join(split_dir, str(img_name).replace("/", "_"))]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")



Found 1 Open_with_CoT.csv test files for Abdomen:
  /content/Abdomen_Test_with_CoT/test/Open_with_CoT.csv

Total open-ended test rows for Abdomen (with valid CoT): 153


In [ ]:
# ---------------------------------------------------------------------------
# 3. System prompt (same as your closed-question / Kaggle version)
# ---------------------------------------------------------------------------
def systemPrompt(question, cot):
    return f"""
              Context:
                     You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

              Objective:
                      Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

              Inputs:
              Question: {question}

              You have to think step by step.

              Instructions:
              1. Examine the medical image carefully.
              2. Independently determine the relevant visual findings before considering the CoT.
              3. Compare your own observations with the provided CoT.
              4. If the CoT is inconsistent with the image, disregard it.
              5. Answer the question using the following evidence priority:
                1. Medical image (highest priority)
                2. User question
                3. CoT (only if verified by the image)
              6. Never fabricate findings or rely on assumptions.

              Output Requirements:
              - Return ONLY the final answer in 1 to 4 words. Do not explain.
              Reasoning Reference:
              {cot}

              Use it only if it agrees with the image.
            """


In [ ]:
# ---------------------------------------------------------------------------
# 4. Model registry -- ONLY ungated models (no HF login/token needed).
#    Matches your closed-question script's 4 models, with Gemma4-E4B
#    swapping in for the 31B variant (confirmed too big for a T4).
#    MedGemma dropped entirely -- it's gated and needs a token.
# ---------------------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

def _load_generic(repo):
    processor = AutoProcessor.from_pretrained(repo)
    model = AutoModelForImageTextToText.from_pretrained(
        repo, quantization_config=bnb_config, device_map="auto",
    )
    return model, processor

def load_llama_vision():
    return _load_generic("unsloth/Llama-3.2-11B-Vision-Instruct")

def load_qwen25_vl():
    return _load_generic("Qwen/Qwen2.5-VL-7B-Instruct")

def load_qwen3_vl():
    return _load_generic("Qwen/Qwen3-VL-8B-Instruct")

def load_gemma4():
    return _load_generic("google/gemma-4-E4B-it")

MODEL_REGISTRY = {
    "Llama-11B":     load_llama_vision,
    "Qwen3-VL-8B":   load_qwen3_vl,
    "Gemma4-E4B":    load_gemma4,
    "Qwen2.5-VL-7B": load_qwen25_vl,
}

REPO_IDS = {
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
    "Gemma4-E4B": "google/gemma-4-E4B-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
}

def generate_free_text(model, processor, image, prompt_text, max_new_tokens=16):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


In [ ]:
# ---------------------------------------------------------------------------
# 5. Run one model, one condition (with/without CoT), over open_df
# ---------------------------------------------------------------------------
def run_condition_open(model, processor, use_cot: bool):
    results = []
    for i, row in open_df.iterrows():
        try:
            img_path = resolve_image_path(row["split_dir"], row[OPEN_IMG_COL])
            image = Image.open(img_path).convert("RGB")
            cot_text = row[COT_COL] if use_cot else ""
            prompt_text = systemPrompt(row["question"], cot_text)
            raw_output = generate_free_text(model, processor, image, prompt_text)
        except Exception as e:
            print(f"    [WARN] row {i} failed: {e}")
            raw_output = ""

        gold = str(row["answer"]).strip()
        results.append({"question": row["question"], "gold": gold, "pred": raw_output})

        if i % 20 == 0:
            print(f"    [{i}/{len(open_df)}] gold={gold!r} pred={raw_output!r}")
    return results



In [ ]:
# ---------------------------------------------------------------------------
# 6. Generation-quality metrics: BLEU, ROUGE-1/2/L, BERTScore P/R/F1
# ---------------------------------------------------------------------------
def compute_generation_metrics(golds, preds):
    bleu_scores = []
    for g, p in zip(golds, preds):
        ref_tokens = [g.lower().split()]
        hyp_tokens = p.lower().split()
        if len(hyp_tokens) == 0:
            bleu_scores.append(0.0)
            continue
        bleu_scores.append(sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=_smoothing))
    bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0

    r1, r2, rl = [], [], []
    for g, p in zip(golds, preds):
        scores = _rouge.score(g, p)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)
    rouge1 = sum(r1) / len(r1) if r1 else 0.0
    rouge2 = sum(r2) / len(r2) if r2 else 0.0
    rougeL = sum(rl) / len(rl) if rl else 0.0

    P, R, F1 = bertscore_score(preds, golds, lang="en", verbose=False)
    bert_p = P.mean().item()
    bert_r = R.mean().item()
    bert_f1 = F1.mean().item()

    return {
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge1, 4),
        "ROUGE-2": round(rouge2, 4),
        "ROUGE-L": round(rougeL, 4),
        "BERTScore F1": round(bert_f1, 4),
        "BERTScore P": round(bert_p, 4),
        "BERTScore R": round(bert_r, 4),
    }


In [ ]:
# ---------------------------------------------------------------------------
# 7. Disk-cache cleanup between models
# ---------------------------------------------------------------------------
import shutil

def clear_model_cache(repo_id: str):
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    folder_name = "models--" + repo_id.replace("/", "--")
    path = os.path.join(cache_dir, folder_name)
    if os.path.exists(path):
        size_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(path) for f in files
        ) / (1024**3)
        shutil.rmtree(path, ignore_errors=True)
        print(f"Cleared cache for {repo_id} (freed ~{size_gb:.1f} GB)")
    else:
        print(f"No cache found for {repo_id} (nothing to clear)")



In [ ]:
# ---------------------------------------------------------------------------
# 8. Run ALL models x BOTH conditions, save results to /content
# ---------------------------------------------------------------------------

# Define OUTPUT_DIR and create it if it doesn't exist
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

open_summary_rows = []
all_open_results = {}

for model_name, loader_fn in MODEL_REGISTRY.items():
    print(f"\n{'='*70}\nLoading {model_name}  [Abdomen - OPEN]\n{'='*70}")
    try:
        model, processor = loader_fn()
    except Exception as e:
        print(f"  [SKIPPING {model_name}] failed to load: {e}")
        clear_model_cache(REPO_IDS[model_name])
        continue

    try:
        for condition, use_cot in [("With", True), ("Without", False)]:
            print(f"\n--- {model_name} | CoT: {condition} | Dataset: Abdomen | OPEN ---")
            results = run_condition_open(model, processor, use_cot)
            all_open_results[f"{model_name}_{condition}"] = results

            golds = [r["gold"] for r in results]
            preds = [r["pred"] for r in results]

            metrics = compute_generation_metrics(golds, preds)
            print(f"Metrics -- {model_name} ({condition} CoT, Abdomen): {metrics}")

            open_summary_rows.append({
                "Dataset": "Abdomen",
                "Model Name": model_name,
                "CoT": condition,
                **metrics,
            })

            pd.DataFrame(open_summary_rows).to_csv(
                f"{OUTPUT_DIR}/open_comparison_summary_Abdomen.csv", index=False
            )
            with open(f"{OUTPUT_DIR}/open_comparison_raw_results_Abdomen.json", "w") as f:
                json.dump(all_open_results, f, indent=2)

    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
        clear_model_cache(REPO_IDS[model_name])


Loading Llama-11B  [Abdomen - OPEN]


preprocessor_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics -- Qwen2.5-VL-7B (With CoT, Abdomen): {'BLEU': 0.1863, 'ROUGE-1': 0.8388, 'ROUGE-2': 0.2434, 'ROUGE-L': 0.8388, 'BERTScore F1': 0.9715, 'BERTScore P': 0.9735, 'BERTScore R': 0.9697}

--- Qwen2.5-VL-7B | CoT: Without | Dataset: Abdomen | OPEN ---
    [0/153] gold='CT' pred='CT scan'
    [20/153] gold='Right Lung' pred='Liver'
    [40/153] gold='Transverse  Plane' pred='Transverse plane'
    [60/153] gold='Transverse  Plane' pred='Transverse plane'
    [80/153] gold='Pulmonary bronchus' pred='Lungs'
    [100/153] gold='Abdomen' pred='Abdomen'
    [120/153] gold='Abdomen' pred='Thorax'
    [140/153] gold='Spinal cord' pred='Brain'


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics -- Qwen2.5-VL-7B (Without CoT, Abdomen): {'BLEU': 0.0908, 'ROUGE-1': 0.433, 'ROUGE-2': 0.0719, 'ROUGE-L': 0.4308, 'BERTScore F1': 0.922, 'BERTScore P': 0.9271, 'BERTScore R': 0.9175}
Cleared cache for Qwen/Qwen2.5-VL-7B-Instruct (freed ~30.9 GB)


In [ ]:
# ---------------------------------------------------------------------------
# 9. Final summary table
# ---------------------------------------------------------------------------
open_summary_df = pd.DataFrame(open_summary_rows)
print("\n" + "=" * 100)
print("OPEN-ENDED SUMMARY TABLE -- Abdomen -- BLEU/ROUGE/BERTScore, all models, both conditions")
print("=" * 100)
print(open_summary_df.to_string(index=False))

print(f"\nSaved: {OUTPUT_DIR}/open_comparison_summary_Abdomen.csv")
print(f"Saved: {OUTPUT_DIR}/open_comparison_raw_results_Abdomen.json")




OPEN-ENDED SUMMARY TABLE -- Abdomen -- BLEU/ROUGE/BERTScore, all models, both conditions
Dataset    Model Name     CoT   BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
Abdomen     Llama-11B    With 0.0718   0.8435   0.2159   0.8435        0.9658       0.9729       0.9593
Abdomen     Llama-11B Without 0.0238   0.3618   0.0196   0.3618        0.9160       0.9261       0.9071
Abdomen   Qwen3-VL-8B    With 0.1799   0.7971   0.2455   0.7945        0.9627       0.9636       0.9619
Abdomen   Qwen3-VL-8B Without 0.0745   0.3775   0.0131   0.3775        0.9056       0.9049       0.9069
Abdomen    Gemma4-E4B    With 0.1040   0.4700   0.1312   0.4700        0.9162       0.9155       0.9179
Abdomen    Gemma4-E4B Without 0.0340   0.2093   0.0109   0.2093        0.8826       0.8784       0.8880
Abdomen Qwen2.5-VL-7B    With 0.1863   0.8388   0.2434   0.8388        0.9715       0.9735       0.9697
Abdomen Qwen2.5-VL-7B Without 0.0908   0.4330   0.0719   0.4308        0.9220 